# Qwen3-VL-8B-Instruct FP16 on Kaggle T4×2 GPU — Production Showcase

English:

This notebook is the reviewer-facing, reproducible production showcase for running **Qwen/Qwen3-VL-8B-Instruct** in native **FP16** on exactly **two NVIDIA Tesla T4 GPUs**. It does not reimplement the model runtime inside notebook cells: the notebook clones the Git repository, starts the repository's FastAPI service, and exercises the model through the same REST endpoints a real client would use.

The showcase is designed to prove four things at once:

- the model is really resident across **GPU T4 ×2** in `torch.float16` with no CPU/disk weight offload;
- the repository can be cloned and re-run by another Kaggle user;
- the REST service works for text, vision, and generic schema-driven extraction;
- the model's capabilities are visible through a deep 19-case multimodal showcase rather than a single smoke test.

Cloudflare exposure is optional. The safe committed default is **`ENABLE_CLOUDFLARE_TUNNEL=False`** (`default: False`), so a normal Run All remains localhost-only.

Tiếng Việt:

Notebook này là production showcase có thể tái kiểm chứng, dành cho reviewer, để chạy **Qwen/Qwen3-VL-8B-Instruct** ở **FP16 nguyên bản** trên đúng **hai GPU NVIDIA Tesla T4**. Notebook không triển khai lại model runtime trong từng cell; thay vào đó notebook clone Git repository, khởi động FastAPI service của chính repository và gọi model thông qua cùng các REST endpoint mà client thực tế sẽ dùng.

Showcase được thiết kế để chứng minh đồng thời bốn điểm:

- model thật sự được phân bổ trên **GPU T4 ×2** ở `torch.float16`, không offload weight xuống CPU/disk;
- người khác có thể clone repository và chạy lại notebook trên Kaggle;
- REST service hoạt động cho text, vision và generic schema-driven extraction;
- năng lực model được thể hiện bằng 19 case multimodal có chiều sâu, không chỉ bằng một smoke test đơn giản.

Cloudflare chỉ là tùy chọn. Giá trị an toàn được commit mặc định là **`ENABLE_CLOUDFLARE_TUNNEL=False`**, vì vậy một lần Run All bình thường chỉ phục vụ localhost.

### What this notebook does not claim / Notebook không tuyên bố

This is a reproducible production-style demo, not a hosted SLA/HA service or a multi-user load benchmark. / Đây là production-style demo có thể tái lập, không phải dịch vụ SLA/HA được host thường trực và cũng không phải benchmark tải multi-user.

## 1 — Reproduce this notebook / Tái kiểm chứng notebook

English:

### Before you run / Trước khi chạy

In Kaggle Notebook settings:

1. Select **Accelerator → GPU T4 x2**.
2. Keep **Internet = ON** so the notebook can clone the public Git repository and, if explicitly requested, create a Cloudflare Quick Tunnel.
3. **Attach the Kaggle Model** `dangkhoa2016/qwen-qwen3-vl-8b-instruct`, framework/variation `Transformers / default / 1`.
4. Keep the repository source ref at the committed default while reviewing a release candidate; an exact tag or commit should be used for final publication authority.
5. Use **Restart Session → Run All** for a clean reproducibility run.

The next code cell locates or clones `Qwen3-VL-8B-Instruct-FP16-On-Kaggle-T4x2-GPU`, records `REPO_REF`, resolves the exact Git HEAD, and switches the notebook working directory to the repository. The model itself remains attached read-only through Kaggle Inputs rather than being downloaded a second time.

Tiếng Việt:

Trong phần cài đặt Kaggle Notebook:

1. Chọn **Accelerator → GPU T4 x2**.
2. Giữ **Internet = ON** để notebook có thể clone Git repository công khai và, chỉ khi chủ động yêu cầu, tạo Cloudflare Quick Tunnel.
3. **Attach Kaggle Model** `dangkhoa2016/qwen-qwen3-vl-8b-instruct`, framework/variation `Transformers / default / 1`.
4. Giữ source ref của repository theo mặc định đã commit khi review release candidate; bản publish cuối nên dùng tag hoặc exact commit bất biến làm authority.
5. Dùng **Restart Session → Run All** để có một lần chạy tái kiểm chứng sạch.

Code cell tiếp theo sẽ tìm hoặc `git clone` repository `Qwen3-VL-8B-Instruct-FP16-On-Kaggle-T4x2-GPU`, ghi `REPO_REF`, resolve exact Git HEAD và chuyển working directory sang repository. Model vẫn được attach read-only qua Kaggle Inputs thay vì download checkpoint lần hai.

### Expected evidence / Kết quả cần thấy

- repository root printed from `/kaggle/working`;
- resolved Git HEAD / exact source identity;
- attached model path exists;
- no fallback download of another full checkpoint.

In [ ]:
from pathlib import Path
import os, sys, subprocess, time, json, html
REPO_URL = "https://github.com/dangkhoa2016/Qwen3-VL-8B-Instruct-FP16-On-Kaggle-T4x2-GPU.git"
REPO_NAME = "Qwen3-VL-8B-Instruct-FP16-On-Kaggle-T4x2-GPU"
REPO_REF = os.getenv("QWEN3VL_REPO_REF", "main")
MODEL_PATH = Path("/kaggle/input/models/dangkhoa2016/qwen-qwen3-vl-8b-instruct/transformers/default/1")
REPO_ROOT = Path("/kaggle/working") / REPO_NAME
# Equivalent shell workflow: git clone ...
if not (REPO_ROOT / ".git").is_dir():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_ROOT)], check=True)
# Equivalent shell workflow: git checkout REPO_REF
subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", REPO_REF], check=True)
assert MODEL_PATH.is_dir(), f"Attach the Kaggle Model first: missing {MODEL_PATH}"
os.chdir(REPO_ROOT); sys.path.insert(0, str(REPO_ROOT / "src"))
REPO_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("REPO_URL=", REPO_URL); print("REPO_REF=", REPO_REF); print("REPO_COMMIT=", REPO_COMMIT); print("MODEL_PATH=", MODEL_PATH)


## 2 — Hardware & software authority / Xác minh phần cứng và môi trường

English:

Before model loading, this gate verifies the live Kaggle accelerator profile. The notebook intentionally fails closed unless CUDA is available, exactly two GPUs are visible, and both device names are Tesla T4. It also records the interpreter and core runtime package versions so the executed artifact is auditable later.

This section is deliberately separate from model identity: hardware authority answers *where the notebook is running*; `/v1/model` later proves *how the model was actually placed*.

Tiếng Việt:

Trước khi load model, gate này xác minh profile accelerator thật của Kaggle session. Notebook cố ý fail-closed nếu CUDA không khả dụng, không có đúng hai GPU hoặc một trong hai device không phải Tesla T4. Cell cũng ghi phiên bản Python và các runtime package chính để artifact đã chạy có thể audit lại sau này.

Section này tách biệt với model identity: hardware authority trả lời *notebook đang chạy ở đâu*; endpoint `/v1/model` phía sau mới chứng minh *model thực sự được đặt như thế nào*.

### Expected evidence / Kết quả cần thấy

- `gpu_count: 2`;
- both names equal `Tesla T4`;
- visible Python, PyTorch, Transformers, Accelerate, FastAPI and Uvicorn versions.

In [ ]:
import torch, transformers, accelerate, fastapi, uvicorn
assert torch.cuda.is_available(), "CUDA is required"
assert torch.cuda.device_count() == 2, f"Expected exactly 2 GPUs, got {torch.cuda.device_count()}"
gpu_names=[torch.cuda.get_device_name(i) for i in range(2)]
assert all("T4" in n for n in gpu_names), f"Expected Tesla T4 ×2, got {gpu_names}"
snapshot={"python":sys.version.split()[0],"torch":torch.__version__,"transformers":transformers.__version__,"accelerate":accelerate.__version__,"fastapi":fastapi.__version__,"uvicorn":uvicorn.__version__,"cuda":torch.version.cuda,"gpu_count":2,"gpu_names":gpu_names,"repository_commit":REPO_COMMIT}
print(json.dumps(snapshot, indent=2))


## 3 — Configuration: localhost first, Cloudflare optional / Cấu hình: ưu tiên localhost, Cloudflare tùy chọn

English:

The production-style default is intentionally conservative: the REST server listens locally and the public tunnel is disabled. The notebook sets the attached model path, server port and **`ENABLE_CLOUDFLARE_TUNNEL=False`** unless you override them before execution.

If you later set the tunnel flag to `True`, the repository requires `QWEN3VL_API_KEY`; this prevents accidentally exposing an anonymous inference endpoint to the public Internet. The secret value is never printed by the notebook.

Tiếng Việt:

Mặc định production-style được giữ an toàn: REST server chỉ lắng nghe local và public tunnel bị tắt. Notebook thiết lập model path đã attach, server port và **`ENABLE_CLOUDFLARE_TUNNEL=False`** trừ khi bạn chủ động override trước khi chạy.

Nếu sau đó bật tunnel thành `True`, repository bắt buộc phải có `QWEN3VL_API_KEY`; điều này tránh vô tình mở anonymous inference endpoint ra Internet. Notebook không bao giờ in giá trị secret.

### Expected evidence / Kết quả cần thấy

- `ENABLE_CLOUDFLARE_TUNNEL=False` on a normal run;
- `QWEN3VL_PORT=8000` unless intentionally changed;
- no API key value printed.

In [ ]:
os.environ.setdefault("QWEN3VL_MODEL_PATH", str(MODEL_PATH))
os.environ.setdefault("QWEN3VL_PORT", "8000")
os.environ.setdefault("ENABLE_CLOUDFLARE_TUNNEL", "False")
print("QWEN3VL_MODEL_PATH=", os.environ["QWEN3VL_MODEL_PATH"])
print("QWEN3VL_PORT=", os.environ["QWEN3VL_PORT"])
print("ENABLE_CLOUDFLARE_TUNNEL=", os.environ["ENABLE_CLOUDFLARE_TUNNEL"])


## 4 — Verify repository-controlled showcase assets / Xác minh bộ asset do repository kiểm soát

English:

The showcase does not depend on ad-hoc images downloaded during execution. `scripts/verify_asset_pack.py` validates the repository-controlled asset set and its case manifest before any expensive inference begins. This keeps prompts, assets and expected evidence stable across reruns.

The current pack contains **23 generated assets** and **19 canonical showcase cases**. A failure here indicates repository/data drift and should stop the notebook before GPU time is spent.

Tiếng Việt:

Showcase không phụ thuộc vào ảnh tải ngẫu nhiên trong lúc chạy. `scripts/verify_asset_pack.py` xác minh bộ asset do repository kiểm soát và case manifest trước khi bắt đầu inference tốn GPU. Nhờ vậy prompt, asset và expected evidence giữ ổn định giữa các lần chạy lại.

Bộ hiện tại có **23 generated assets** và **19 canonical showcase cases**. Nếu gate này fail thì đó là dấu hiệu repository/data bị lệch và notebook nên dừng trước khi tiêu tốn GPU.

### Expected evidence / Kết quả cần thấy

`ASSET_PACK_STATIC_VERIFY=PASS generated=23 cases=19`

In [ ]:
subprocess.run([sys.executable, "scripts/verify_asset_pack.py", "."], check=True)


## 5 — Start the FastAPI REST server / Khởi động FastAPI REST server

English:

This is the central architecture of the notebook. The model is loaded **once** by `scripts/start_server.py`; every later demo request goes through HTTP instead of calling the Python model object directly. This proves the repository's service layer and avoids reloading an ~8B multimodal model for every capability section.

The notebook waits on `/health` until the service reports both `model_loaded=True` and `ready=True`. If the server exits early, the tail of the server log is surfaced as the notebook error so failure is diagnosable.

Tiếng Việt:

Đây là kiến trúc trung tâm của notebook. Model chỉ được load **một lần** bởi `scripts/start_server.py`; mọi demo phía sau đều gọi qua HTTP thay vì gọi trực tiếp Python model object. Cách này vừa chứng minh service layer của repository, vừa tránh reload model multimodal ~8B cho từng section.

Notebook poll `/health` cho tới khi service báo đồng thời `model_loaded=True` và `ready=True`. Nếu server chết sớm, phần cuối server log được đưa thẳng vào lỗi notebook để có thể chẩn đoán.

### Expected evidence / Kết quả cần thấy

- `SERVER_PID=<pid>`;
- `/health` eventually returns HTTP 200;
- `SERVER_READY=PASS`.

In [ ]:
import httpx
PORT=int(os.environ["QWEN3VL_PORT"]); BASE_URL=f"http://127.0.0.1:{PORT}"; LOG_PATH=REPO_ROOT/"server.log"
server_proc=subprocess.Popen([sys.executable,"scripts/start_server.py"],stdout=LOG_PATH.open("w"),stderr=subprocess.STDOUT,env=os.environ.copy())
print("SERVER_PID=",server_proc.pid)
ready=False
with httpx.Client(timeout=2.0) as client:
    for _ in range(240):
        if server_proc.poll() is not None: raise RuntimeError(LOG_PATH.read_text(errors="replace")[-8000:])
        try:
            r=client.get(f"{BASE_URL}/health")
            if r.status_code==200 and r.json().get("ready") is True: ready=True; break
        except httpx.HTTPError: pass
        time.sleep(1)
assert ready, "REST server did not become ready"
print("SERVER_READY=PASS", BASE_URL)


## 6 — Runtime identity: prove FP16 dual-T4 placement / Chứng minh model FP16 thật sự nằm trên hai T4

English:

Hardware presence alone is not enough. The `/v1/model` endpoint exposes reviewer-safe runtime identity and device placement after the model has actually loaded. This gate requires `torch.float16`, exactly two Tesla T4 devices, and explicitly rejects CPU or disk weight offload.

The returned `device_map` also shows the real split: the vision stack and early language layers reside on GPU 0 while the remaining language layers and head reside on GPU 1. This is Hugging Face big-model dispatch/model sharding, not data parallelism or pooled 32 GB VRAM.

Tiếng Việt:

Chỉ thấy hai GPU là chưa đủ. Endpoint `/v1/model` cung cấp runtime identity và device placement an toàn cho reviewer sau khi model đã load thật. Gate này bắt buộc `torch.float16`, đúng hai Tesla T4 và từ chối rõ ràng mọi CPU/disk weight offload.

`device_map` trả về cũng cho thấy cách chia thật: vision stack và các language layer đầu nằm trên GPU 0, phần language layer còn lại và head nằm trên GPU 1. Đây là Hugging Face big-model dispatch/model sharding, không phải data parallel và cũng không phải một vùng VRAM 32 GB hợp nhất.

### Expected evidence / Kết quả cần thấy

- `dtype = torch.float16`;
- `gpu_count = 2` and both GPU names are `Tesla T4`;
- `cpu_weight_offload = false`;
- `disk_weight_offload = false`;
- `RUNTIME_IDENTITY_GATE=PASS`.

In [ ]:
with httpx.Client(base_url=BASE_URL,timeout=30.0) as client:
    health=client.get("/health").json(); model_info=client.get("/v1/model").json()
assert health=={"status":"ok","model_loaded":True,"ready":True}
assert model_info["dtype"]=="torch.float16" and model_info["gpu_count"]==2
assert model_info["gpu_names"]==["Tesla T4","Tesla T4"]
assert model_info["cpu_weight_offload"] is False and model_info["disk_weight_offload"] is False
print(json.dumps(model_info,indent=2,ensure_ascii=False)); print("RUNTIME_IDENTITY_GATE=PASS")


## 7 — REST API Examples / Ví dụ REST API

English:

Before the full benchmark, this section demonstrates how an external client talks to the service. The API is intentionally specialized rather than a single overloaded route:

- `GET /health` — readiness;
- `GET /v1/model` — runtime identity;
- `POST /v1/generate` — text-only generation;
- `POST /v1/vision` — image + prompt via multipart upload;
- `POST /v1/extract` — image + generic JSON Schema via multipart upload;
- `GET /v1/metrics` — request and GPU runtime metrics.

The text smoke request must return `QWEN3VL_T4X2_OK`. The notebook also prints copyable `curl` examples for `/v1/vision` and `/v1/extract`; image input is multipart file upload only—no `image_url` or base64 API contract is required.

Tiếng Việt:

Trước full benchmark, section này minh họa cách client bên ngoài giao tiếp với service. API cố ý dùng các endpoint chuyên biệt thay vì gom mọi hành vi vào một route:

- `GET /health` — readiness;
- `GET /v1/model` — runtime identity;
- `POST /v1/generate` — text-only generation;
- `POST /v1/vision` — ảnh + prompt bằng multipart upload;
- `POST /v1/extract` — ảnh + generic JSON Schema bằng multipart upload;
- `GET /v1/metrics` — request và GPU runtime metrics.

Text smoke phải trả về `QWEN3VL_T4X2_OK`. Notebook cũng in ví dụ `curl` có thể copy cho `/v1/vision` và `/v1/extract`; image input chỉ dùng multipart file upload, không cần contract `image_url` hay base64.

### Expected evidence / Kết quả cần thấy

- HTTP 200 from `/v1/generate`;
- exact marker `TEXT_API_GATE=PASS`;
- visible, copyable multipart examples for `/v1/vision` and `/v1/extract`.

In [ ]:
with httpx.Client(base_url=BASE_URL,timeout=120.0) as client:
    r=client.post("/v1/generate",json={"prompt":"Reply with exactly: QWEN3VL_T4X2_OK","max_new_tokens":32,"temperature":0.0}); r.raise_for_status(); text_result=r.json()
print(json.dumps(text_result,indent=2,ensure_ascii=False)); assert "QWEN3VL_T4X2_OK" in text_result["output"]; print("TEXT_API_GATE=PASS")


In [ ]:
vision_curl = """curl -X POST http://127.0.0.1:8000/v1/vision \
  -F 'image=@examples/realistic/bustling_waterfront_produce_market.jpg' \
  -F 'prompt=Describe the market scene and read visible price signs.'"""
extract_curl = """curl -X POST http://127.0.0.1:8000/v1/extract \
  -F 'image=@examples/realistic/northstar_office_supply_invoice.jpg' \
  -F 'schema={"type":"object","properties":{"vendor":{"type":"string"}}}'"""
print("POST /v1/vision\n" + vision_curl)
print("\nPOST /v1/extract\n" + extract_curl)


## 8 — Run the canonical showcase exactly once / Chạy canonical showcase đúng một lần

English:

This is the only cell that executes the full 19-case multimodal workload. `scripts/run_showcase.py` sends every case through the already-running REST server, writes machine-readable JSONL/summary evidence, and `scripts/verify_evidence.py` checks the resulting run.

**Important:** all later capability sections only re-render these recorded results. They do not repeat inference merely to create a nicer notebook. This keeps GPU usage bounded and guarantees that the dashboard, galleries and final scorecard all refer to the same run.

Automatic semantic checks are intentionally limited to the five structured extraction cases, where JSON parsing, schema validity and expected field values can be checked deterministically. Free-form vision answers remain reviewer-inspected evidence rather than being auto-passed from HTTP success alone.

Tiếng Việt:

Đây là cell duy nhất chạy toàn bộ workload multimodal 19 case. `scripts/run_showcase.py` gửi từng case qua REST server đang chạy, ghi evidence JSONL/summary dạng machine-readable và `scripts/verify_evidence.py` xác minh kết quả của run đó.

**Quan trọng:** mọi capability section phía sau chỉ render lại kết quả đã ghi. Notebook không inference lặp lại chỉ để hiển thị đẹp hơn. Nhờ vậy GPU usage được giới hạn và dashboard, gallery, final scorecard đều tham chiếu đúng cùng một run.

Automatic semantic check chỉ áp dụng cho năm structured extraction case, nơi JSON parse, schema validity và expected field values có thể kiểm tra xác định. Các câu trả lời vision free-form vẫn là evidence để reviewer đọc, không tự động PASS chỉ vì HTTP thành công.

### Expected evidence / Kết quả cần thấy

- exactly 19 recorded cases;
- `EVIDENCE_VERIFY=PASS cases=19 runtime_pass=19` on a successful run;
- only one invocation of `scripts/run_showcase.py` in the notebook.

In [ ]:
from IPython.display import display, Markdown, Image as IPyImage, HTML
subprocess.run([sys.executable,"scripts/run_showcase.py","--base-url",BASE_URL,"--output-dir","results"],check=True)
subprocess.run([sys.executable,"scripts/verify_evidence.py"],check=True)
RESULTS_JSONL=REPO_ROOT/"results"/"showcase-results.jsonl"
showcase_records=[json.loads(x) for x in RESULTS_JSONL.read_text(encoding="utf-8").splitlines() if x.strip()]
assert len(showcase_records)==19
print("SHOWCASE_CASES_RECORDED=19")


In [ ]:
def _pretty(v): return json.dumps(v,ensure_ascii=False,indent=2) if not isinstance(v,str) else v
def show_case(record):
    semantic=record.get("semantic_validation") or {}; sem=semantic.get("pass"); semantic_label="PASS" if sem is True else ("FAIL" if sem is False else "REVIEW")
    latency=record.get("wall_latency_ms") or 0
    display(HTML(f"<div style='border:1px solid #ddd;border-radius:12px;padding:14px 18px;margin:18px 0'><h3 style='margin:0 0 8px'>{html.escape(record['case_id'])}</h3><b>Endpoint:</b> {html.escape(record.get('endpoint',''))} &nbsp; | &nbsp; <b>Latency:</b> {latency:,} ms &nbsp; | &nbsp; <b>Runtime:</b> {'PASS' if record.get('runtime_success') else 'FAIL'} &nbsp; | &nbsp; <b>Semantic:</b> {semantic_label}</div>"))
    asset=REPO_ROOT/record["asset"]
    if asset.is_file(): display(IPyImage(filename=str(asset),width=760))
    req=record.get("request",{})
    if req.get("prompt"): display(Markdown("**Task / Prompt**")); print(req["prompt"])
    if req.get("schema"): display(Markdown("**Requested JSON Schema**")); print(_pretty(req["schema"]))
    response=record.get("response",{}); display(Markdown("**Model Response**")); print(_pretty(response.get("output",response.get("data",response))))
    if record.get("expected") is not None: display(Markdown("**Expected / Ground Truth**")); print(_pretty(record["expected"]))
    display(Markdown("**Evaluation**")); print(_pretty(semantic))
def show_cases(ids):
    idx={r['case_id']:r for r in showcase_records}
    for cid in ids: show_case(idx[cid])


# Capability Showcase / Trình diễn năng lực model

English:

The remaining sections are presentation views over the single canonical result set created above. Each case card shows the input image, task/prompt, endpoint, model response, expected evidence when available, **Latency**, runtime verdict and **Semantic** verdict. The goal is to let a reviewer judge what the model can actually do without reading raw JSONL files.

Tiếng Việt:

Các section còn lại là lớp trình bày dựa trên cùng một canonical result set đã tạo phía trên. Mỗi case card hiển thị ảnh đầu vào, task/prompt, endpoint, model response, expected evidence nếu có, **Latency**, runtime verdict và **Semantic** verdict. Mục tiêu là để reviewer nhìn trực tiếp model làm được gì mà không phải tự đọc file JSONL thô.

## 9 — General Visual Understanding / Hiểu hình ảnh tổng quát

English:

These photorealistic scene cases test broad visual comprehension: recognizing objects, reading scene context, summarizing what is happening and connecting multiple visible details into a coherent answer. They are useful for judging whether the model behaves like a practical visual assistant rather than a narrow OCR engine.

Because these answers are free-form, the notebook presents the real model response and expected review context but does not fabricate an automatic semantic score.

Tiếng Việt:

Các case cảnh photorealistic kiểm tra khả năng hiểu hình ảnh tổng quát: nhận diện vật thể, đọc bối cảnh, tóm tắt điều đang xảy ra và liên kết nhiều chi tiết nhìn thấy thành câu trả lời mạch lạc. Đây là cách đánh giá model như một visual assistant thực dụng chứ không chỉ là OCR engine hẹp.

Vì câu trả lời mang tính free-form, notebook hiển thị nguyên model response và expected review context nhưng không bịa ra automatic semantic score.

### Expected evidence / Kết quả cần thấy

Reviewer should see the source image, prompt, full model answer, HTTP/runtime PASS and per-case latency. / Reviewer cần thấy ảnh nguồn, prompt, câu trả lời đầy đủ, HTTP/runtime PASS và latency của từng case.

In [ ]:
show_cases(["vision-office-rich-01","vision-market-photorealistic-01"])


## 10 — Spatial & Relationship Reasoning / Suy luận không gian và quan hệ

English:

These synthetic scenes are constructed to require more than object naming. Questions ask about relative position, adjacency, left/right relationships and object-to-object relations. This section therefore probes whether the model can reason over the spatial organization of a scene.

Tiếng Việt:

Các cảnh synthetic này được thiết kế để yêu cầu nhiều hơn việc chỉ gọi tên vật thể. Câu hỏi tập trung vào vị trí tương đối, adjacency, quan hệ trái/phải và quan hệ giữa các vật thể. Vì vậy section này kiểm tra khả năng suy luận dựa trên tổ chức không gian của toàn cảnh.

### Expected evidence / Kết quả cần thấy

The rendered answer should explicitly address the requested relationships, while the case card retains the original prompt and image for manual verification. / Câu trả lời phải xử lý rõ các quan hệ được hỏi, đồng thời case card giữ nguyên prompt và ảnh để reviewer đối chiếu thủ công.

In [ ]:
show_cases(["vision-parking-spatial-01","vision-cafe-relations-01"])


## 11 — OCR & Document Understanding / OCR và hiểu tài liệu

English:

Document capability is evaluated as *understanding*, not just transcription. The cases include notices, warehouse summaries, invoices and receipts where the model must read text and then answer a question, combine fields or perform simple reasoning over the document content.

This separation matters: a model may OCR individual tokens correctly but still fail to understand what a notice requires or what an invoice implies.

Tiếng Việt:

Năng lực tài liệu được đánh giá ở mức *hiểu nội dung*, không chỉ transcription. Các case gồm notice, warehouse summary, invoice và receipt; model phải đọc chữ rồi trả lời câu hỏi, kết hợp nhiều field hoặc thực hiện suy luận đơn giản từ nội dung tài liệu.

Sự phân biệt này quan trọng vì một model có thể OCR đúng từng token nhưng vẫn không hiểu notice yêu cầu gì hoặc invoice thể hiện điều gì.

### Expected evidence / Kết quả cần thấy

Visible source document + question + model answer + latency + runtime verdict, allowing direct human comparison against the document. / Phải thấy trực tiếp tài liệu nguồn + câu hỏi + model answer + latency + runtime verdict để reviewer đối chiếu với tài liệu.

In [ ]:
show_cases(["document-transit-qa-01","document-warehouse-qa-01","invoice-understanding-01","receipt-understanding-01"])


## 12 — Structured JSON Extraction / Trích xuất JSON có cấu trúc

English:

`POST /v1/extract` is schema-driven rather than document-type-specific. The caller supplies a JSON Schema together with the image, so the same endpoint can handle an invoice, receipt, business card, service form or Vietnamese delivery note without adding a new route for each document type.

For these five cases the notebook can evaluate semantics automatically. A PASS requires all three layers: valid JSON parse, valid JSON Schema, and the expected subset of extracted field values. HTTP 200 by itself is not sufficient.

Tiếng Việt:

`POST /v1/extract` hoạt động theo schema thay vì hard-code theo loại tài liệu. Client gửi JSON Schema cùng ảnh, vì vậy cùng một endpoint có thể xử lý invoice, receipt, business card, service form hoặc Vietnamese delivery note mà không cần tạo route riêng cho từng loại.

Với năm case này notebook có thể semantic-evaluate tự động. PASS phải đồng thời đạt ba lớp: JSON parse hợp lệ, JSON Schema hợp lệ và expected subset của field value phải khớp. Chỉ HTTP 200 là chưa đủ.

### Expected evidence / Kết quả cần thấy

For every structured case: source image, requested schema, model JSON, `JSON parse ✓`, `Schema valid ✓`, and expected-field semantic PASS. / Với từng structured case: ảnh nguồn, schema yêu cầu, JSON model trả về, `JSON parse ✓`, `Schema valid ✓` và expected-field semantic PASS.

In [ ]:
show_cases(["invoice-extract-01","receipt-extract-01","business-card-extract-01","form-extract-01"])


## 13 — Tables & Visual Data Reasoning / Bảng và suy luận dữ liệu trực quan

English:

Table cases test row/column lookup plus reasoning across cells—for example comparing quarterly values or delivery-performance metrics. The model must correctly associate headers, rows and numbers before it can answer the question.

Tiếng Việt:

Các table case kiểm tra cả việc tra cứu row/column lẫn suy luận giữa nhiều cell—ví dụ so sánh số liệu theo quý hoặc delivery-performance metrics. Model phải gắn đúng header, row và number trước khi có thể trả lời câu hỏi.

### Expected evidence / Kết quả cần thấy

The answer should cite the correct table values and conclusion, with the source table rendered immediately above it for manual checking. / Câu trả lời cần dùng đúng số liệu trong bảng và đưa ra kết luận đúng; bảng nguồn được render ngay phía trên để reviewer kiểm tra.

In [ ]:
show_cases(["table-sales-01","table-delivery-01"])


## 14 — Charts & Numerical Reasoning / Biểu đồ và suy luận số liệu

English:

Chart cases evaluate visual series reading, maxima/minima, trend interpretation and simple arithmetic comparisons. They are intentionally separated from tables because chart understanding requires mapping visual marks and axes to quantitative values before reasoning.

Tiếng Việt:

Chart case kiểm tra khả năng đọc visual series, tìm cực đại/cực tiểu, diễn giải xu hướng và thực hiện so sánh số học đơn giản. Chúng được tách khỏi table vì hiểu chart đòi hỏi model ánh xạ mark/axis trực quan sang giá trị định lượng trước khi suy luận.

### Expected evidence / Kết quả cần thấy

Reviewer should be able to compare the plotted values against the model's numerical answer and reasoning directly in the notebook. / Reviewer phải có thể đối chiếu trực tiếp giá trị trên biểu đồ với câu trả lời số liệu và reasoning của model ngay trong notebook.

In [ ]:
show_cases(["chart-tickets-01","chart-satisfaction-01"])


## 15 — Vietnamese & Multilingual / Tiếng Việt và ngữ cảnh đa ngôn ngữ

English:

These cases verify that the vision-language path remains useful when Vietnamese appears in the image, in the requested schema, or together with English instructions. The workload includes both free-form question answering and structured Vietnamese extraction.

The goal is not to claim exhaustive language coverage; it is to provide concrete reproducible evidence that the same REST service can handle representative Vietnamese/mixed-language tasks.

Tiếng Việt:

Các case này xác minh vision-language path vẫn hữu ích khi tiếng Việt xuất hiện trong ảnh, trong schema yêu cầu hoặc kết hợp với instruction tiếng Anh. Workload gồm cả free-form question answering và structured extraction tiếng Việt.

Mục tiêu không phải tuyên bố bao phủ ngôn ngữ toàn diện; đây là evidence cụ thể và có thể tái lập rằng cùng một REST service xử lý được các task tiếng Việt/mixed-language đại diện.

### Expected evidence / Kết quả cần thấy

At least one structured Vietnamese extraction must pass automatic semantic validation, while mixed-language free-form cases remain reviewer-visible. / Ít nhất một structured Vietnamese extraction phải PASS semantic validation tự động; các mixed-language free-form case vẫn hiển thị để reviewer tự đánh giá.

In [ ]:
show_cases(["vi-delivery-extract-01","mixed-workshop-qa-01","vi-notice-en-question-01"])


## 16 — Performance & Stability / Hiệu năng và độ ổn định service

English:

After the capability workload, `scripts/run_stability.py` sends 12 sequential requests across `/v1/generate`, `/v1/vision` and `/v1/extract` while reusing the same loaded model. The test records latency and GPU allocator deltas to catch obvious service-state growth or identity changes.

This **12/12 service stability gate is not the original 20-run baseline acceptance campaign**. It is a repository/API-layer regression check for this production showcase. Keeping those two claims separate prevents the notebook from overstating what was measured.

Tiếng Việt:

Sau workload năng lực, `scripts/run_stability.py` gửi 12 request tuần tự qua `/v1/generate`, `/v1/vision` và `/v1/extract` trong khi tái sử dụng đúng model đã load. Test ghi latency và GPU allocator delta để phát hiện service-state growth hoặc model identity thay đổi bất thường.

**Gate service stability 12/12 này không phải campaign baseline acceptance 20-run ban đầu.** Đây là regression check ở lớp repository/API dành cho production showcase. Tách hai claim này giúp notebook không diễn giải quá mức dữ liệu đã đo.

### Expected evidence / Kết quả cần thấy

- `12/12` requests PASS;
- `server_model_identity_stable=true`;
- allocated VRAM delta remains zero or otherwise explicitly reported;
- a separate median/max stability latency summary.

In [ ]:
subprocess.run([sys.executable,"scripts/run_stability.py","--base-url",BASE_URL,"--requests","12"],check=True)
stability=json.loads((REPO_ROOT/"results"/"stability-summary.json").read_text()); print(json.dumps(stability,indent=2,ensure_ascii=False))


## 17 — Results Dashboard / Dashboard kết quả

English:

The dashboard condenses all 19 recorded cases into one reviewer-oriented table. It is a navigation aid—not a replacement for the individual case cards. Runtime success, automatic structured semantics and manual/free-form review remain distinct columns so a successful HTTP response is never confused with semantic correctness.

Tiếng Việt:

Dashboard gom toàn bộ 19 case đã ghi thành một bảng dễ review. Đây là công cụ điều hướng, không thay thế các case card chi tiết. Runtime success, automatic structured semantics và manual/free-form review được giữ thành các cột riêng để HTTP thành công không bị nhầm với semantic correctness.

### Expected evidence / Kết quả cần thấy

- exactly 19 rows/cases represented;
- each case shows capability group, endpoint, runtime result, semantic status and latency;
- no second inference run is triggered by rendering the dashboard.

In [ ]:
import pandas as pd
rows=[]
for r in showcase_records:
    sem=(r.get("semantic_validation") or {}).get("pass")
    rows.append({"case":r["case_id"],"group":r["group"],"endpoint":r["endpoint"],"runtime":"PASS" if r["runtime_success"] else "FAIL","semantic":"PASS" if sem is True else ("FAIL" if sem is False else "REVIEW"),"latency_ms":r.get("wall_latency_ms")})
df=pd.DataFrame(rows); display(df)
print("Runtime PASS:",(df.runtime=="PASS").sum(),"/",len(df)); print("Automatic Semantic PASS:",(df.semantic=="PASS").sum()); print("Manual-review cases:",(df.semantic=="REVIEW").sum())


## 18 — Final Scorecard / Bảng điểm cuối

English:

The final scorecard is the concise acceptance view for this notebook run. Read each number according to its evidence class:

- **19/19 runtime** — every canonical case completed successfully through REST;
- **5/5 structured semantic** — all auto-evaluable extraction cases passed parse, schema and expected-field checks;
- **free-form cases** — shown for reviewer inspection rather than auto-scored;
- **12/12 stability** — sequential service-layer requests reused one stable model instance;
- latency and GPU metrics — describe this exact executed run, not a universal performance guarantee.

Tiếng Việt:

Final scorecard là góc nhìn acceptance ngắn gọn cho chính notebook run này. Mỗi con số phải được hiểu theo đúng lớp evidence:

- **19/19 runtime** — toàn bộ canonical case hoàn thành thành công qua REST;
- **5/5 structured semantic** — mọi extraction case có thể auto-evaluate đều PASS parse, schema và expected-field check;
- **free-form cases** — hiển thị để reviewer kiểm tra thay vì tự động chấm điểm;
- **12/12 stability** — request tuần tự ở service layer tái sử dụng cùng một model instance ổn định;
- latency và GPU metrics — mô tả chính run đã execute này, không phải bảo đảm hiệu năng phổ quát.

### Expected evidence / Kết quả cần thấy

`REST_API_GATE=PASS` and `FINAL_SHOWCASE_RUNTIME=PASS`, together with visible counts and latency/VRAM summary. / Cần thấy `REST_API_GATE=PASS`, `FINAL_SHOWCASE_RUNTIME=PASS` cùng các count và tóm tắt latency/VRAM.

In [ ]:
summary=json.loads((REPO_ROOT/"results"/"showcase-summary.json").read_text())
with httpx.Client(base_url=BASE_URL,timeout=30.0) as client: metrics=client.get("/v1/metrics").json()
final={"repository_commit":REPO_COMMIT,"model":model_info["model_id"],"dtype":model_info["dtype"],"gpu_names":model_info["gpu_names"],"cpu_weight_offload":model_info["cpu_weight_offload"],"disk_weight_offload":model_info["disk_weight_offload"],"showcase_runtime":f"{summary['showcase_runtime_pass']}/{summary['showcase_cases_total']}","structured_parse_rate":summary["structured_parse_rate"],"structured_schema_valid_rate":summary["structured_schema_valid_rate"],"structured_semantic":f"{summary['semantic_auto_pass']}/{summary['semantic_auto_evaluated']}","stability":f"{stability['requests_pass']}/{stability['requests_total']}","latency_median_ms":metrics.get("latency_ms",{}).get("median"),"latency_p95_ms":metrics.get("latency_ms",{}).get("p95"),"cloudflare_enabled":os.environ.get("ENABLE_CLOUDFLARE_TUNNEL","False")}
display(pd.DataFrame([final]).T.rename(columns={0:"value"}))
assert summary["showcase_runtime_fail"]==0 and summary["structured_parse_rate"]==1.0 and summary["structured_schema_valid_rate"]==1.0
assert stability["service_stability_gate"]=="PASS"
print("REST_API_GATE=PASS"); print("FINAL_SHOWCASE_RUNTIME=PASS")


## 19 — Optional authenticated Cloudflare Quick Tunnel / Cloudflare Quick Tunnel có xác thực, tùy chọn

English:

The notebook is fully complete without a public URL. To expose the REST API temporarily, deliberately set:

```python
os.environ["ENABLE_CLOUDFLARE_TUNNEL"] = "True"
os.environ["QWEN3VL_API_KEY"] = "<secret>"
```

Then launch `scripts/start_cloudflare_tunnel.sh`. The helper refuses to create a tunnel when the API key is absent. Public clients must send `Authorization: Bearer <API_KEY>`; localhost remains the preferred path for notebook verification.

Quick Tunnels are ephemeral review/demo transport, not a production hosting commitment. Do not publish the API key in notebook outputs.

Tiếng Việt:

Notebook đã hoàn tất đầy đủ mà không cần public URL. Nếu muốn tạm expose REST API, hãy chủ động đặt:

```python
os.environ["ENABLE_CLOUDFLARE_TUNNEL"] = "True"
os.environ["QWEN3VL_API_KEY"] = "<secret>"
```

Sau đó chạy `scripts/start_cloudflare_tunnel.sh`. Helper sẽ từ chối tạo tunnel nếu thiếu API key. Public client phải gửi `Authorization: Bearer <API_KEY>`; localhost vẫn là đường ưu tiên để verify notebook.

Quick Tunnel chỉ là transport tạm thời cho review/demo, không phải cam kết production hosting. Không publish API key trong notebook output.

### Expected evidence / Kết quả cần thấy

With the committed default, tunnel remains disabled. When intentionally enabled with a secret, the helper prints a temporary `trycloudflare.com` URL without exposing the key. / Với mặc định đã commit, tunnel phải tắt; khi chủ động bật kèm secret, helper in URL `trycloudflare.com` tạm thời mà không lộ key.

## 20 — Cleanup / Dọn dẹp session

English:

The final helper stops only the REST server process created by this notebook. Use it when interactive review is finished or before manually restarting the server in the same Kaggle session. Generated evidence files remain under `results/` so they can still be inspected or exported.

If you intentionally launched a Cloudflare Quick Tunnel, stop that tunnel separately before ending the session.

Tiếng Việt:

Helper cuối chỉ dừng REST server process do chính notebook này tạo. Hãy dùng nó sau khi review tương tác xong hoặc trước khi chủ động restart server trong cùng Kaggle session. Các evidence file đã tạo vẫn được giữ trong `results/` để tiếp tục kiểm tra hoặc export.

Nếu bạn đã chủ động bật Cloudflare Quick Tunnel thì cần dừng tunnel đó riêng trước khi kết thúc session.

### Expected evidence / Kết quả cần thấy

Calling `stop_showcase_server()` prints `SERVER_STOPPED` and does not delete showcase evidence. / Khi gọi `stop_showcase_server()`, notebook in `SERVER_STOPPED` và không xóa showcase evidence.

In [ ]:
def stop_showcase_server():
    if server_proc.poll() is None:
        server_proc.terminate()
        try: server_proc.wait(timeout=15)
        except subprocess.TimeoutExpired: server_proc.kill(); server_proc.wait()
    print("SERVER_STOPPED")
print("Cleanup helper ready: stop_showcase_server()")
